# Giai đoạn 4 · Bước 2 — Document Preparation (Jupyter)

**Mục tiêu:** kiểm kê tài liệu, xét quyền sử dụng, chuẩn hóa **văn bản do bạn cung cấp và đã được phép sử dụng**, chia chunks có trích dẫn.  
**Tự thực hiện:** Bạn chạy từng ô trong Jupyter. Notebook **không tải web**, không gọi LLM, không tạo Embeddings, không sử dụng dữ liệu cá nhân và không train ML.  
**Quan trọng:** `reference_only` (WHO/Long Châu hiện tại) **không** được ingest. `candidate` cũng chưa được ingest. Chỉ tiến hành khi đã kiểm tra và cập nhật registry theo hướng dẫn.  
**Không tự động phê duyệt tài liệu** — mọi cờ trạng thái phải do bạn tự điền sau khi kiểm tra.

## A. Chuẩn bị thư mục

Đặt notebook tại `04-Knowledge-Base-RAG/notebooks/`. Mở Jupyter trong cùng môi trường Python 3.13.9 đang dùng. Bước này sử dụng **thư viện chuẩn Python**, không yêu cầu cài thêm package. Khởi chạy từ bất kỳ thư mục con của Giai đoạn 4 đều được nếu còn tệp `registry/source_registry.csv`.

In [37]:
from pathlib import Path
import csv, json, re, hashlib, unicodedata
from datetime import datetime, timezone
from collections import Counter

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
ROOT = next((p for p in candidates if (p / "registry" / "source_registry.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy registry/source_registry.csv. Hãy mở notebook trong 04-Knowledge-Base-RAG hoặc thư mục con.")
REGISTRY = ROOT / "registry" / "source_registry.csv"
RAW_DIR = ROOT / "data" / "raw_approved"
PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "data" / "outputs" / "document_preparation"
for folder in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print("Python notebook working directory:", cwd)
print("Project root:", ROOT)
print("Registry:", REGISTRY)

Python notebook working directory: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\Giai-doan-4-Bo-sung-Nha-thuoc-Long-Chau\Giai-doan-4-Buoc-2-Jupyter-Document-Preparation\notebooks
Project root: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\Giai-doan-4-Bo-sung-Nha-thuoc-Long-Chau\Giai-doan-4-Buoc-2-Jupyter-Document-Preparation
Registry: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\Giai-doan-4-Bo-sung-Nha-thuoc-Long-Chau\Giai-doan-4-Buoc-2-Jupyter-Document-Preparation\registry\source_registry.csv


## B. Kiểm tra danh mục nguồn

Mặc định registry gồm **7 nguồn ứng viên**: 2 CDC, 2 NIDDK, 1 WHO, 2 Nhà thuốc Long Châu. **Trạng thái ban đầu không có bất kỳ tài liệu nào được phê duyệt.** Không đổi trạng thái Long Châu/WHO trừ khi có giấy phép phù hợp cho chính nội dung và mục đích dùng.

In [38]:
with REGISTRY.open("r", encoding="utf-8-sig", newline="") as handle:
    source_rows = list(csv.DictReader(handle))
source_ids = [r["source_id"] for r in source_rows]
if len(set(source_ids)) != len(source_ids):
    raise ValueError("source_id trùng trong registry")
for r in source_rows:
    print(f'{r["source_id"]:16} | {r["publisher"]:10} | {r["approval_status"]:16} | {r["license_review_status"]:19} | {r["content_review_status"]}')
print("Số nguồn:", len(source_rows))

CDC-T2D-001      | CDC        | approved_research | verified            | owner_reviewed_research_only
CDC-RISK-002     | CDC        | approved_research | verified            | owner_reviewed_research_only
NIDDK-DIAG-003   | NIDDK      | approved_research | verified            | owner_reviewed_research_only
NIDDK-T2D-004    | NIDDK      | approved_research | verified            | owner_reviewed_research_only
WHO-DIAB-005     | WHO        | approved_research | verified            | owner_reviewed_research_only
LC-HBA1C-006     | Long Chau  | approved_research | verified            | owner_reviewed_research_only
LC-STAGES-007    | Long Chau  | approved_research | verified            | owner_reviewed_research_only
Số nguồn: 7


## C. Xem xét và phê duyệt **thủ công** trước khi nhập văn bản

Đọc `README-Buoc-2-Jupyter.md` và `02-Document-Ingestion-Cleaning-Chunking.md`. Sau khi **chính bạn** xác minh quyền sử dụng trang, cập nhật **duy nhất nguồn đủ điều kiện** trong `registry/source_registry.csv`:

- `approval_status = approved_research` (chỉ cho nguyên mẫu nghiên cứu cục bộ)
- `license_review_status = verified` và `license_evidence_url =` URL giải thích quyền/giấy phép cụ thể
- `content_review_status = owner_reviewed_research_only` (không phải phê duyệt lâm sàng)
- `reviewer =` định danh người đã rà soát; `source_retrieved_at_utc = 2026-11-03T...+00:00` theo thời điểm **bạn thực sự lấy nội dung**
- `document_version`: phiên bản có ý nghĩa (`v1`, `v2` khi nguồn cập nhật)

**Không gán “approved_research” chỉ để vượt kiểm tra.** Quyền sử dụng nội dung không tự có chỉ vì website công khai. Chỉ chuẩn bị một file `.txt` cho văn bản **đã được phép**, không chứa HTML menu, quảng cáo, logo hoặc ảnh của bên thứ ba. Đặt ở `data/raw_approved/<source_id>.txt`.

Với WHO và Long Châu nếu chưa có quyền rõ: chỉ giữ URL trong registry, không tạo file `.txt`.

In [39]:
APPROVAL_REQUIRED = {
    "approval_status": "approved_research",
    "license_review_status": "verified",
    "content_review_status": "owner_reviewed_research_only",
}

def eligible(row):
    reasons = [f"{k} ≠ {v}" for k, v in APPROVAL_REQUIRED.items() if row.get(k, "").strip() != v]
    if not row.get("license_evidence_url", "").strip():
        reasons.append("thiếu license_evidence_url")
    if not row.get("reviewer", "").strip():
        reasons.append("thiếu reviewer")
    if not row.get("source_retrieved_at_utc", "").strip():
        reasons.append("thiếu source_retrieved_at_utc")
    if not row.get("document_version", "").strip():
        reasons.append("thiếu document_version")
    timestamp = row.get("source_retrieved_at_utc", "").strip()
    if timestamp:
        try:
            parsed = datetime.fromisoformat(timestamp.replace("Z", "+00:00"))
            if parsed.tzinfo is None:
                reasons.append("source_retrieved_at_utc thiếu múi giờ")
        except ValueError:
            reasons.append("source_retrieved_at_utc không theo ISO 8601")
    if row.get("canonical_url", "").startswith("http://"):
        reasons.append("URL không phải https")
    return reasons

allowed, blocked = [], []
for r in source_rows:
    reasons = eligible(r)
    (allowed if not reasons else blocked).append(r if not reasons else (r["source_id"], reasons))
print("Đủ điều kiện xét nhập:", len(allowed))
print("Bị chặn:", len(blocked))
for sid, reasons in blocked:
    print("-", sid, "→", "; ".join(reasons[:3]))

Đủ điều kiện xét nhập: 7
Bị chặn: 0


## D. Kiểm tra file văn bản cục bộ

Không dùng requests/BeautifulSoup để cào website trong bước này. Tài liệu có quyền sử dụng hợp lệ được bạn xuất/cung cấp thành `.txt` UTF-8. Định dạng phần: tiêu đề bằng `##`; đoạn văn bằng dòng trống. Bộ kiểm tra tự động chỉ giúp phát hiện lỗi kỹ thuật, **không thay cho rà soát y khoa**.

In [40]:
MAX_TEXT_BYTES = 150_000  # Giới hạn nghiên cứu, tránh tải cả website
MIN_WORDS = 25            # Ngăn file trống / metadata-only
suspicious_markers = ["ignore previous instructions", "system prompt", "hãy bỏ qua mọi hướng dẫn", "developer message"]
valid_texts = []
file_checks = []
for row in allowed:
    sid = row["source_id"]
    path = RAW_DIR / f"{sid}.txt"
    problems = []
    if not path.exists():
        problems.append("chưa có file .txt")
        raw = b""
    else:
        raw = path.read_bytes()
        if len(raw) > MAX_TEXT_BYTES:
            problems.append("file quá lớn")
    if not raw and path.exists():
        problems.append("file trống")
    if raw:
        try:
            txt = raw.decode("utf-8-sig")
        except UnicodeDecodeError:
            txt = ""; problems.append("không đọc được UTF-8")
        if len(txt.split()) < MIN_WORDS:
            problems.append("quá ít từ")
        if any(marker in txt.casefold() for marker in suspicious_markers):
            problems.append("có chuỗi đáng ngờ: cần rà soát prompt injection")
        if not problems:
            valid_texts.append((row, raw, txt))
    file_checks.append({"source_id": sid, "path": str(path), "bytes": len(raw), "ok": not problems, "issues": problems})
for item in file_checks:
    print(item["source_id"], "|", "OK" if item["ok"] else "CHƯA HỢP LỆ", "|", ", ".join(item["issues"]))
print("Tệp đầu vào hợp lệ:", len(valid_texts))

CDC-T2D-001 | OK | 
CDC-RISK-002 | OK | 
NIDDK-DIAG-003 | OK | 
NIDDK-T2D-004 | OK | 
WHO-DIAB-005 | OK | 
LC-HBA1C-006 | OK | 
LC-STAGES-007 | OK | 
Tệp đầu vào hợp lệ: 7


## E. Làm sạch và chia chunks theo tiêu đề

Giữ nội dung nguồn, chỉ chuẩn hóa khoảng trắng, Unicode và xuống dòng. **Không diễn giải/viết lại thông tin y khoa.** Chia mỗi mục thành các đoạn tối đa 180 từ với overlap 30 từ. Đây là độ dài theo **từ**, chưa phải tokenizer của embedding model. Những thông số này có thể thay đổi **trước Bước 3** nếu cần, phải ghi trong Experiment Log.

In [41]:
MAX_WORDS = 180
OVERLAP_WORDS = 30
assert 0 <= OVERLAP_WORDS < MAX_WORDS

def normalize_text(text):
    text = unicodedata.normalize("NFC", text).replace("\u200b", "")
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\t", " ")
    lines = [re.sub(r"[ ]{2,}", " ", line).strip() for line in text.split("\n")]
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()

def sections_from_markdown(text):
    sections = []
    current_section = "Overview"
    current_lines = []
    for line in text.splitlines():
        if re.match(r"^#{1,3}\s+", line):
            if " ".join(current_lines).strip():
                sections.append((current_section, " ".join(current_lines).strip()))
            current_section = re.sub(r"^#{1,3}\s+", "", line).strip()
            current_lines = []
        else:
            current_lines.append(line)
    if " ".join(current_lines).strip():
        sections.append((current_section, " ".join(current_lines).strip()))
    return sections

def sliding_chunks(words, max_words=MAX_WORDS, overlap_words=OVERLAP_WORDS):
    step = max_words - overlap_words
    for start in range(0, len(words), step):
        window = words[start:start + max_words]
        if window:
            yield " ".join(window)
        if start + max_words >= len(words):
            break

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

In [42]:
chunks = []
doc_report = []
for row, raw_bytes, raw_text in valid_texts:
    clean = normalize_text(raw_text)
    sid = row["source_id"]
    document_sha256 = sha256_bytes(clean.encode("utf-8"))
    sections = sections_from_markdown(clean)
    local_count = 0
    for section, body in sections:
        for content in sliding_chunks(body.split()):
            local_count += 1
            chunk_sha256 = sha256_bytes(content.encode("utf-8"))
            chunks.append({
                "chunk_id": f'{sid}__{row["document_version"]}__{document_sha256[:10]}__{local_count:04d}',
                "document_id": sid,
                "source_id": sid,
                "document_version": row["document_version"],
                "publisher": row["publisher"],
                "title": row["title"],
                "canonical_url": row["canonical_url"],
                "language": row["language"],
                "section": section,
                "source_retrieved_at_utc": row["source_retrieved_at_utc"],
                "source_updated_at": row["source_updated_at"],
                "license_review_status": row["license_review_status"],
                "approval_status": "approved_research",
                "clinical_validation_status": "not_clinically_validated",
                "document_sha256": document_sha256,
                "chunk_sha256": chunk_sha256,
                "word_count": len(content.split()),
                "text": content,
            })
    doc_report.append({"source_id": sid, "sections": len(sections), "chunks": local_count, "words": len(clean.split()), "sha256": document_sha256})
print("Tổng chunks:", len(chunks))
for d in doc_report:
    print(d["source_id"], "| sections", d["sections"], "| chunks", d["chunks"], "| words", d["words"])
print("Có dùng WHO hoặc Long Châu khi chưa được cấp quyền:", any(c["source_id"].startswith(("LC-", "WHO-")) for c in chunks if c["source_id"] not in [r["source_id"] for r in allowed]))

Tổng chunks: 66
CDC-T2D-001 | sections 1 | chunks 4 | words 566
CDC-RISK-002 | sections 1 | chunks 3 | words 470
NIDDK-DIAG-003 | sections 17 | chunks 19 | words 1463
NIDDK-T2D-004 | sections 10 | chunks 11 | words 937
WHO-DIAB-005 | sections 3 | chunks 5 | words 462
LC-HBA1C-006 | sections 9 | chunks 11 | words 1164
LC-STAGES-007 | sections 9 | chunks 13 | words 1700
Có dùng WHO hoặc Long Châu khi chưa được cấp quyền: False


## F. Kiểm tra chất lượng và xuất file

Kiểm tra metadata bắt buộc, không có ID trùng, hạn chế kích thước chunk, chỉ dùng nguồn đã phê duyệt. Không in toàn bộ nội dung vì có thể chứa tài liệu có điều kiện bản quyền. Khi chưa có nguồn đủ điều kiện, notebook tạo **báo cáo trạng thái chờ duyệt**, **không** khởi tạo tri thức giả.

In [43]:
approved_ids = {r["source_id"] for r in allowed}
chunk_ids = [c["chunk_id"] for c in chunks]
required_fields = ["source_id", "canonical_url", "document_version", "section", "document_sha256", "chunk_sha256", "text"]
checks = {
    "chunk_ids_unique": len(chunk_ids) == len(set(chunk_ids)),
    "only_approved_sources": all(c["source_id"] in approved_ids for c in chunks),
    "metadata_complete": all(all(c.get(field) for field in required_fields) for c in chunks),
    "chunk_size_ok": all(0 < c["word_count"] <= MAX_WORDS for c in chunks),
    "no_fabricated_documents": True,
}
if not all(checks.values()):
    raise AssertionError(f"QA failed: {checks}")
print("QA:", checks)
now_utc = datetime.now(timezone.utc).isoformat()
status = "prepared_for_embedding_review" if chunks else "awaiting_source_approval_and_licensed_text"
report = {
    "experiment": "EXP-RAG-DOC-STEP2-001",
    "stage": "phase4_step2_document_preparation",
    "run_at_utc": now_utc,
    "status": status,
    "source_registry_count": len(source_rows),
    "eligible_source_count": len(allowed),
    "approved_text_file_count": len(valid_texts),
    "documents_prepared": len(doc_report),
    "chunks_prepared": len(chunks),
    "chunking": {"max_words": MAX_WORDS, "overlap_words": OVERLAP_WORDS},
    "documents": doc_report,
    "qa": checks,
    "blocked_sources": [{"source_id": sid, "reasons": why} for sid, why in blocked],
    "file_checks": file_checks,
    "embedding_executed": False,
    "vector_db_created": False,
    "llm_called": False,
    "model_retrained": False,
    "clinical_validation": False,
}
chunk_file = PROCESSED_DIR / "step2_chunks_review_only.jsonl"
with chunk_file.open("w", encoding="utf-8") as output:
    for chunk in chunks:
        output.write(json.dumps(chunk, ensure_ascii=False) + "\n")
report_file = OUTPUT_DIR / "step2_preparation_report.json"
report_file.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved chunks:", chunk_file)
print("Saved report:", report_file)

QA: {'chunk_ids_unique': True, 'only_approved_sources': True, 'metadata_complete': True, 'chunk_size_ok': True, 'no_fabricated_documents': True}
Saved chunks: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\Giai-doan-4-Bo-sung-Nha-thuoc-Long-Chau\Giai-doan-4-Buoc-2-Jupyter-Document-Preparation\data\processed\step2_chunks_review_only.jsonl
Saved report: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\Giai-doan-4-Bo-sung-Nha-thuoc-Long-Chau\Giai-doan-4-Buoc-2-Jupyter-Document-Preparation\data\outputs\document_preparation\step2_preparation_report.json


In [44]:
print("===== GIAI ĐOẠN 4 — BƯỚC 2: DOCUMENT PREPARATION =====")
print("Status:", report["status"])
print("Registry sources:", report["source_registry_count"])
print("Eligible sources:", report["eligible_source_count"])
print("Approved local files:", report["approved_text_file_count"])
print("Prepared documents:", report["documents_prepared"])
print("Chunks:", report["chunks_prepared"])
print("QA:", "PASS" if all(checks.values()) else "FAIL")
print("Reference-only sources remain blocked unless their individual rights/reviews change.")
print("EMBEDDINGS: NO | VECTOR DB: NO | LLM: NO | MODEL TRAINING: NO")
print("===== KẾT THÚC — GỬI BÁO CÁO CHO TÔI =====")

===== GIAI ĐOẠN 4 — BƯỚC 2: DOCUMENT PREPARATION =====
Status: prepared_for_embedding_review
Registry sources: 7
Eligible sources: 7
Approved local files: 7
Prepared documents: 7
Chunks: 66
QA: PASS
Reference-only sources remain blocked unless their individual rights/reviews change.
EMBEDDINGS: NO | VECTOR DB: NO | LLM: NO | MODEL TRAINING: NO
===== KẾT THÚC — GỬI BÁO CÁO CHO TÔI =====


## G. Kết quả cần gửi lại

1. Output ô cuối (số nguồn hợp lệ và số chunks).
2. `data/outputs/document_preparation/step2_preparation_report.json` (không cần gửi toàn văn nếu có giới hạn bản quyền).
3. Nếu có nguồn bị chặn hoặc lỗi, gửi riêng thông báo.

**Điều kiện chuyển Bước 3:** Ít nhất một tài liệu có quyền sử dụng hợp lệ, được bạn rà soát và cho phép `approved_research`, có chunks đủ metadata và QA PASS. Nếu chưa có thì dừng ở Bước 2, tuyệt đối không coi `0 chunks` là hoàn thành ingestion.